# 02 — Auxotrophy benchmark reproduction

**Workflow version:** 0.5.3

Reproduce the 147 gene-compound phenotype benchmark using COBRApy and GLPK. The reference candidate now uses process-isolated fresh SBML loading: every phenotype is simulated in a new Python process, that process loads the released SBML once, performs one knockout/rescue test, returns one JSON result, and exits. This avoids the model-state failures observed with `Model.copy()`, sequential model contexts, and JSON reconstruction.

The previous 0.5.2 `pristine_copy` outputs are provisional. They are replaced as reference outputs only after both 147-pair process-isolated runs complete successfully.

## Notebook navigation convention

Every code cell starts with a stable **Code Cell number**, **Requires**, and **Output/Provides** comment. Use these labels instead of Jupyter execution counters.


In [ ]:
# Code Cell 1 — Setup imports, protocol constants, and paths
# Requires: nothing
# Provides: protocol constants and repository paths

from pathlib import Path
import hashlib
import json
import math
import platform
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import cobra
from cobra.io import read_sbml_model

WORKFLOW_VERSION = "0.5.3"
SOLVER = "glpk"
UPTAKE_LOWER_BOUND = -1000.0
VIABILITY_FRACTION = 0.01
ISOLATION_MODE = "fresh_process_sbml"
PROCESS_TIMEOUT_SECONDS = 180

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = ROOT / "data" / "raw"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

ORIGINAL_XML = DATA_DIR / "yeast9.0.xml"
CURATED_XML = DATA_DIR / "Yeast9_curated.xml"
DATASET_XLSX = DATA_DIR / "mmc3.xlsx"
PROCESS_WORKER = ROOT / "scripts" / "fresh_pair_worker.py"

ALIASES = {
    "Yeast9": {},
    "Yeast9_curated": {"a_0001": "r_temp1"},
}

if not PROCESS_WORKER.exists():
    raise FileNotFoundError(f"Missing process worker: {PROCESS_WORKER}")


In [ ]:
# Code Cell 2 — Parse Dataset 2 into stable phenotype records
# Requires: Code Cell 1
# Provides: pairs with 147 benchmark records

REQUIRED_COLUMNS = [
    "Gene Systematic Name", "Chemical", "exchange", "ID",
    "Strain Background", "Reference",
]

def split_plus(value):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("+") if part.strip()]

def split_genes(value):
    if pd.isna(value):
        return []
    return [
        part.strip()
        for part in re.split(r"\s+and\s+", str(value).strip(), flags=re.I)
        if part.strip()
    ]

def count_exchange_targets(value):
    if pd.isna(value):
        return 0
    text = str(value).strip()
    return len(re.findall(r"\bexchange\b", text, flags=re.I))

def parse_dataset(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="all")
    missing = [column for column in REQUIRED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing Dataset 2 columns: {missing}")

    records = []
    for idx, row in raw.iterrows():
        excel_row = idx + 2
        genes = split_genes(row["Gene Systematic Name"])
        ids = split_plus(row["ID"])
        chemical = str(row["Chemical"]).strip()
        n_targets = count_exchange_targets(row["exchange"])
        conditional = bool(re.search(r"\badd\b", chemical, flags=re.I))

        if not genes or not ids or n_targets < 1 or len(ids) < n_targets:
            raise ValueError(f"Cannot parse Dataset 2 Excel row {excel_row}.")

        rescue_ids = ids[:n_targets] if conditional else ids
        background_ids = ids[n_targets:] if conditional else []

        records.append({
            "pair_id": f"excel:{excel_row}",
            "excel_row": excel_row,
            "gene_field": str(row["Gene Systematic Name"]).strip(),
            "genes": genes,
            "n_genes": len(genes),
            "chemical": chemical,
            "exchange_field": str(row["exchange"]).strip(),
            "id_field": str(row["ID"]).strip(),
            "rescue_ids": rescue_ids,
            "background_ids": background_ids,
            "conditional_medium": conditional,
            "strain_background": str(row["Strain Background"]).strip(),
            "reference": str(row["Reference"]).strip(),
        })

    pairs = pd.DataFrame(records)

    if len(pairs) != 147:
        raise AssertionError(f"Expected 147 records, found {len(pairs)}.")

    if pairs["pair_id"].duplicated().any():
        raise AssertionError("pair_id values are not unique.")

    return pairs

pairs = parse_dataset(DATASET_XLSX)

display(pairs.head())
print("Records:", len(pairs))
print("Conditional-medium records:", int(pairs["conditional_medium"].sum()))
print("Multi-gene records:", int((pairs["n_genes"] > 1).sum()))


In [ ]:
# Code Cell 3 — Run one phenotype in a fresh Python process
# Requires: Code Cells 1–2
# Provides: run_fresh_process_pair and provenance helpers

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

def get_git_commit(root: Path) -> str:
    completed = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=root,
        capture_output=True,
        text=True,
        check=True,
    )
    return completed.stdout.strip()

def format_duration(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours:02d}h {minutes:02d}m {seconds:02d}s"
    return f"{minutes:02d}m {seconds:02d}s"

def format_growth(value):
    if value is None:
        return "nan"

    value = float(value)
    if not np.isfinite(value):
        return "nan"

    return f"{value:.6g}"

def parse_worker_json(stdout: str, pair_id: str) -> dict:
    lines = [line.strip() for line in stdout.splitlines() if line.strip()]

    for line in reversed(lines):
        try:
            payload = json.loads(line)
        except json.JSONDecodeError:
            continue

        if isinstance(payload, dict) and payload.get("pair_id") == pair_id:
            return payload

    raise RuntimeError(f"Worker returned no valid JSON result for {pair_id}.")

def run_fresh_process_pair(
    model_path,
    model_label,
    record,
    threshold,
    aliases,
):
    command = [
        sys.executable,
        str(PROCESS_WORKER),
        "--model-path",
        str(model_path),
        "--model-label",
        model_label,
        "--solver",
        SOLVER,
        "--threshold",
        repr(float(threshold)),
        "--uptake-lower-bound",
        repr(float(UPTAKE_LOWER_BOUND)),
        "--aliases-json",
        json.dumps(aliases),
        "--pair-id",
        record["pair_id"],
    ]

    for gene_id in record["genes"]:
        command.extend(["--gene", gene_id])

    for reaction_id in record["background_ids"]:
        command.extend(["--background", reaction_id])

    for reaction_id in record["rescue_ids"]:
        command.extend(["--rescue", reaction_id])

    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=PROCESS_TIMEOUT_SECONDS,
            check=False,
        )
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            f"Fresh-process worker timed out for {record['pair_id']} "
            f"after {PROCESS_TIMEOUT_SECONDS} seconds."
        ) from exc

    if completed.returncode != 0:
        raise RuntimeError(
            f"Fresh-process worker failed for {record['pair_id']}.\n"
            f"stdout:\n{completed.stdout}\n"
            f"stderr:\n{completed.stderr}"
        )

    result = parse_worker_json(completed.stdout, record["pair_id"])

    result.update({
        "excel_row": record["excel_row"],
        "gene_field": record["gene_field"],
        "chemical": record["chemical"],
        "n_genes": record["n_genes"],
        "strain_background": record["strain_background"],
        "reference": record["reference"],
        "conditional_medium": record["conditional_medium"],
    })

    expected_hash = sha256_file(Path(model_path))
    if result.get("model_sha256") != expected_hash:
        raise RuntimeError(
            f"Worker model hash mismatch for {record['pair_id']}."
        )

    return result


In [ ]:
# Code Cell 4 — Run a resumable process-isolated benchmark
# Requires: Code Cells 1–3
# Provides: benchmark_signature and run_benchmark

def benchmark_signature(model_path, model_label, threshold, git_commit):
    payload = {
        "workflow_version": WORKFLOW_VERSION,
        "workflow_commit": git_commit,
        "model": model_label,
        "model_sha256": sha256_file(model_path),
        "dataset_sha256": sha256_file(DATASET_XLSX),
        "worker_sha256": sha256_file(PROCESS_WORKER),
        "python": sys.version,
        "cobra": cobra.__version__,
        "solver": SOLVER,
        "threshold": float(threshold),
        "uptake_lower_bound": UPTAKE_LOWER_BOUND,
        "mode": ISOLATION_MODE,
        "process_timeout_seconds": PROCESS_TIMEOUT_SECONDS,
    }
    encoded = json.dumps(payload, sort_keys=True).encode()
    return hashlib.sha256(encoded).hexdigest(), payload

def run_benchmark(
    model_path,
    model_label,
    pairs_df,
    threshold,
    aliases,
    git_commit,
    resume=True,
):
    checkpoint_csv = (
        RESULTS_DIR
        / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.csv"
    )
    checkpoint_json = (
        RESULTS_DIR
        / f"02_checkpoint_{model_label}_{ISOLATION_MODE}.json"
    )

    signature, signature_payload = benchmark_signature(
        model_path,
        model_label,
        threshold,
        git_commit,
    )

    completed = pd.DataFrame()
    completed_ids = set()

    if resume and checkpoint_csv.exists() and checkpoint_json.exists():
        metadata = json.loads(
            checkpoint_json.read_text(encoding="utf-8")
        )

        if metadata.get("signature") != signature:
            raise RuntimeError(
                f"Checkpoint protocol mismatch for {model_label}."
            )

        completed = pd.read_csv(checkpoint_csv)
        completed_ids = set(completed["pair_id"].astype(str))

    new_results = []
    recent_times = []
    session_start = time.perf_counter()
    total = len(pairs_df)

    if completed_ids:
        print(
            f"{model_label}: resuming from "
            f"{len(completed_ids)}/{total} completed pairs.",
            flush=True,
        )
    else:
        print(
            f"{model_label}: starting fresh-process benchmark "
            f"with {total} pairs.",
            flush=True,
        )

    for record in pairs_df.to_dict("records"):
        if record["pair_id"] in completed_ids:
            continue

        already_done = len(completed) + len(new_results)

        print(
            f"START {already_done + 1:3d}/{total} | "
            f"{record['pair_id']} | "
            f"{record['gene_field']} | "
            f"{record['chemical']}",
            flush=True,
        )

        pair_start = time.perf_counter()

        result = run_fresh_process_pair(
            model_path,
            model_label,
            record,
            threshold,
            aliases,
        )

        pair_seconds = time.perf_counter() - pair_start
        new_results.append(result)

        current = pd.concat(
            [completed, pd.DataFrame(new_results)],
            ignore_index=True,
        )

        current = (
            current.drop_duplicates(subset=["pair_id"], keep="last")
            .sort_values("excel_row")
            .reset_index(drop=True)
        )

        current.to_csv(checkpoint_csv, index=False)

        checkpoint_metadata = {
            "signature": signature,
            "signature_payload": signature_payload,
            "model": model_label,
            "isolation_mode": ISOLATION_MODE,
            "workflow_version": WORKFLOW_VERSION,
            "workflow_commit": git_commit,
            "completed_pairs": len(current),
        }

        checkpoint_json.write_text(
            json.dumps(checkpoint_metadata, indent=2),
            encoding="utf-8",
        )

        recent_times.append(pair_seconds)
        recent_times = recent_times[-10:]

        done = len(current)
        remaining = total - done
        eta = float(np.mean(recent_times)) * remaining
        elapsed = time.perf_counter() - session_start

        print(
            f"DONE  {done:3d}/{total} | "
            f"{record['pair_id']} | "
            f"class={result['classification']} | "
            f"KO={result['ko_status']}:{format_growth(result['ko_growth'])} | "
            f"rescue={result['rescue_status']}:{format_growth(result['rescue_growth'])} | "
            f"pid={result['pid']} | "
            f"last={format_duration(pair_seconds)} | "
            f"elapsed={format_duration(elapsed)} | "
            f"ETA={format_duration(eta)}",
            flush=True,
        )

    final = pd.concat(
        [completed, pd.DataFrame(new_results)],
        ignore_index=True,
    )

    final = (
        final.drop_duplicates(subset=["pair_id"], keep="last")
        .sort_values("excel_row")
        .reset_index(drop=True)
    )

    if len(final) != total:
        raise AssertionError(
            f"Expected {total} completed pairs for {model_label}, "
            f"found {len(final)}."
        )

    final.to_csv(
        RESULTS_DIR / f"02_{model_label}_pair_results.csv",
        index=False,
    )

    print(
        f"{model_label}: DONE ({len(final)}/{total})",
        flush=True,
    )

    return final


In [ ]:
# Code Cell 5 — Calculate the paper-method viability threshold
# Requires: Code Cells 1–4
# Provides: wt_growth, viability_threshold, RUN_GIT_COMMIT

threshold_model = read_sbml_model(str(ORIGINAL_XML))
threshold_model.solver = SOLVER

wt_growth = threshold_model.slim_optimize(error_value=np.nan)

if str(threshold_model.solver.status).lower() != "optimal":
    raise RuntimeError(
        "Cannot calculate the viability threshold "
        "from an optimal WT solution."
    )

viability_threshold = VIABILITY_FRACTION * float(wt_growth)
RUN_GIT_COMMIT = get_git_commit(ROOT)

print("Workflow version:", WORKFLOW_VERSION)
print("Workflow commit:", RUN_GIT_COMMIT)
print("Isolation mode:", ISOLATION_MODE)
print("WT growth:", wt_growth)
print("Viability threshold:", viability_threshold)
print("Worker SHA-256:", sha256_file(PROCESS_WORKER))


In [ ]:
# Code Cell 6 — Run the 147-pair Yeast9 benchmark
# Requires: Code Cells 1–5
# Output: results/02_Yeast9_pair_results.csv and local checkpoints

original_results = run_benchmark(
    ORIGINAL_XML,
    "Yeast9",
    pairs,
    viability_threshold,
    ALIASES["Yeast9"],
    RUN_GIT_COMMIT,
    resume=True,
)


In [ ]:
# Code Cell 7 — Run the 147-pair curated Yeast9 benchmark
# Requires: Code Cells 1–6
# Output: results/02_Yeast9_curated_pair_results.csv and local checkpoints

curated_results = run_benchmark(
    CURATED_XML,
    "Yeast9_curated",
    pairs,
    viability_threshold,
    ALIASES["Yeast9_curated"],
    RUN_GIT_COMMIT,
    resume=True,
)


In [ ]:
# Code Cell 8 — Summarize, compare, and write the run manifest
# Requires: Code Cells 1–7
# Output: benchmark summary, pairwise comparison, and 02_run_manifest.json

def summarize(results):
    return {
        "n": len(results),
        "correct": int(
            (results["classification"] == "correct").sum()
        ),
        "type_I": int(
            (results["classification"] == "type_I").sum()
        ),
        "type_II": int(
            (results["classification"] == "type_II").sum()
        ),
        "solver_error": int(
            (results["classification"] == "solver_error").sum()
        ),
        "input_error": int(
            (results["classification"] == "input_error").sum()
        ),
    }

summary = pd.DataFrame([
    {"model": "Yeast9", **summarize(original_results)},
    {"model": "Yeast9_curated", **summarize(curated_results)},
])

summary["accuracy"] = summary["correct"] / summary["n"]

if int(summary["solver_error"].sum()) != 0:
    raise RuntimeError("Reference benchmark contains solver_error rows.")

if int(summary["input_error"].sum()) != 0:
    raise RuntimeError("Reference benchmark contains input_error rows.")

comparison = original_results.merge(
    curated_results,
    on="pair_id",
    suffixes=("_original", "_curated"),
    validate="one_to_one",
)

comparison["change"] = np.select(
    [
        (
            comparison["classification_original"] != "correct"
        )
        & (
            comparison["classification_curated"] == "correct"
        ),
        (
            comparison["classification_original"] == "correct"
        )
        & (
            comparison["classification_curated"] != "correct"
        ),
    ],
    ["fixed", "regression"],
    default="unchanged",
)

display(summary)
display(
    comparison["change"]
    .value_counts()
    .rename_axis("change")
    .to_frame("n")
)

summary.to_csv(
    RESULTS_DIR / "02_benchmark_summary.csv",
    index=False,
)

comparison.to_csv(
    RESULTS_DIR / "02_pairwise_comparison.csv",
    index=False,
)

run_manifest = {
    "workflow_version": WORKFLOW_VERSION,
    "workflow_commit": RUN_GIT_COMMIT,
    "python": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "cobra": cobra.__version__,
    "solver": SOLVER,
    "isolation_mode": ISOLATION_MODE,
    "process_timeout_seconds": PROCESS_TIMEOUT_SECONDS,
    "viability_fraction": VIABILITY_FRACTION,
    "viability_threshold": viability_threshold,
    "uptake_lower_bound": UPTAKE_LOWER_BOUND,
    "worker_sha256": sha256_file(PROCESS_WORKER),
    "files": [
        {
            "file": ORIGINAL_XML.name,
            "bytes": ORIGINAL_XML.stat().st_size,
            "sha256": sha256_file(ORIGINAL_XML),
        },
        {
            "file": CURATED_XML.name,
            "bytes": CURATED_XML.stat().st_size,
            "sha256": sha256_file(CURATED_XML),
        },
        {
            "file": DATASET_XLSX.name,
            "bytes": DATASET_XLSX.stat().st_size,
            "sha256": sha256_file(DATASET_XLSX),
        },
    ],
    "summary": json.loads(summary.to_json(orient="records")),
}

(RESULTS_DIR / "02_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2),
    encoding="utf-8",
)


## Interpretation checkpoint

Do not interpret the benchmark if `solver_error` or `input_error` is non-zero. The paper reports 93/147 correct predictions for Yeast9 and 117/147 for the curated model, but those values are comparison targets, not forced expected outputs.

This 0.5.3 workflow treats process-isolated fresh-SBML loading as the reference implementation candidate because the six-case diagnostic reproduced the independently observed fresh-load behavior while using a different process ID for every phenotype. Residual differences from the paper must still be separated into solver, protocol, and artifact-level causes.
